# Mythos Talking Avatar — One-Click Generator**Pipeline:** Script → Speech → Lip-synced video (all automated)**Voice:** en-US-ChristopherNeural (deep, authoritative)**Model:** Wan 2.2 TextImage2Video 5B FastWan**Segments:** 6 clips, each under 15 seconds> Click **Runtime → Run all** then upload your photo when prompted.

---## Step 1 — Check GPU

In [ ]:
import subprocess, torch

try:
    result = subprocess.run(['nvidia-smi'], check=True, capture_output=True, text=True)
    print(result.stdout)
except Exception as exc:
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → GPU → Save.') from exc

print(f'PyTorch CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f'VRAM: {vram:.1f} GB')
    if vram < 12:
        print('WARNING: Low VRAM. Use 480p resolution.')

---## Step 2 — Install all dependencies

In [ ]:
import subprocess, sys, os

print("Installing edge-tts...")
subprocess.run([sys.executable, '-m', 'pip', 'install', 'edge-tts', '-q'], check=True)

print("Installing system deps...")
env = os.environ.copy()
env['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.run(['sudo', 'apt-get', 'update', '-qq'], check=False, env=env)
subprocess.run(['sudo', 'apt-get', 'install', '-y', '--no-install-recommends',
    'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'], check=False, env=env)

print("All dependencies installed.")

---## Step 3 — Generate all 6 audio segments (auto)

In [ ]:
import edge_tts, asyncio, os, subprocess

voice = "en-US-ChristopherNeural"
rate = "+5%"
pitch = "-2Hz"

segments = {
    'seg1a': 'Listen up, MYTHOS just dropped, and everybody out here is acting like it is just another model. BRO, WAKE UP. This thing is not an upgrade. It is a detonation.',
    'seg1b': 'Mythos is the first AI that does not wait for instructions. It hunts for outcomes. You give it a goal, it builds the plan, the steps, the strategy, the execution.',
    'seg1c': 'It is like hiring a Navy SEAL, a CEO, and a supercomputer, for the price of a coffee.',
    'seg2': 'People are scared of it. GOOD. Be scared. Because Mythos is the model that separates the spectators from the dominators.',
    'seg3': 'Everyone else is playing with chatbots. I am telling you, Mythos is a business weapon. You want to 10X your output? You want to dominate your niche? You want to stop being a consumer and start being a MACHINE?',
    'seg4': 'Then stop sleeping on Mythos. This is the moment the whole AI game flips. The ones who move NOW, they are the ones who own the next decade. Get in. Or get left behind.'
}

os.makedirs('/content/audio', exist_ok=True)
print(f'Generating {len(segments)} audio segments...\n')

async def gen():
    for name, script in segments.items():
        wav = f'/content/audio/{name}.wav'
        mp3 = f'/content/audio/{name}.mp3'
        comm = edge_tts.Communicate(script, voice, rate=rate, pitch=pitch)
        await comm.save(mp3)
        subprocess.run(['ffmpeg', '-i', mp3, '-ar', '16000', '-ac', '1', wav, '-y'], capture_output=True)
        r = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                           '-of', 'default=noprint_wrappers=1:nokey=1', wav],
                          capture_output=True, text=True)
        dur = float(r.stdout.strip()) if r.stdout.strip() else 0
        print(f'  {name}: {dur:.1f}s')
    print(f'\nAll {len(segments)} segments saved to /content/audio/')

asyncio.run(gen())

---## Step 4 — Upload your photo

In [ ]:
from google.colab import files
import os

print('Upload your photo (face forward, good lighting):')
uploaded = files.upload()

if not uploaded:
    raise RuntimeError('No file uploaded. Run this cell again.')

PHOTO_PATH = os.path.join('/content', list(uploaded.keys())[0])

from PIL import Image
img = Image.open(PHOTO_PATH)
print(f'\nPhoto saved: {PHOTO_PATH}')
print(f'Size: {img.size[0]}x{img.size[1]}')
display(img)

---## Step 5 — Setup Wan2GP (clone + install)

In [ ]:
import subprocess, os, sys
from pathlib import Path

WAN2GP_ROOT = Path('/content/Wan2GP').resolve()
WAN_DATA_ROOT = Path('/content/Wan2GP-data').resolve()
WAN_CKPTS_DIR = WAN_DATA_ROOT / 'ckpts'
WAN_LORAS_DIR = WAN_DATA_ROOT / 'loras'
WAN_OUTPUTS_DIR = WAN_DATA_ROOT / 'outputs'
WAN_CACHE_DIR = WAN_DATA_ROOT / 'cache'

for d in [WAN_DATA_ROOT, WAN_CKPTS_DIR, WAN_LORAS_DIR, WAN_OUTPUTS_DIR, WAN_CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Clone
if not WAN2GP_ROOT.exists():
    print('Cloning Wan2GP...')
    subprocess.run(['git', 'clone', 'https://github.com/deepbeepmeep/Wan2GP.git', str(WAN2GP_ROOT)], check=True)
else:
    print('Wan2GP already cloned.')

# Symlinks
def safe_link(repo_path, data_path):
    data_path.mkdir(parents=True, exist_ok=True)
    try:
        if repo_path.is_symlink() and repo_path.resolve() == data_path.resolve():
            return
        if repo_path.is_symlink():
            repo_path.unlink()
        if repo_path.is_dir():
            for child in list(repo_path.iterdir()):
                dest = data_path / child.name
                if not dest.exists():
                    import shutil
                    shutil.move(str(child), str(dest))
            try: repo_path.rmdir()
            except: pass
        repo_path.symlink_to(data_path, target_is_directory=True)
        print(f'  Linked {repo_path.name}')
    except Exception as e:
        print(f'  WARNING: {e}')

safe_link(WAN2GP_ROOT / 'ckpts', WAN_CKPTS_DIR)
safe_link(WAN2GP_ROOT / 'loras', WAN_LORAS_DIR)
safe_link(WAN2GP_ROOT / 'outputs', WAN_OUTPUTS_DIR)

# Install Python deps
print('\nInstalling torch...')
env = os.environ.copy()

try:
    import torch
    if torch.cuda.is_available():
        print(f'CUDA torch ready: {torch.__version__}')
    else:
        raise ImportError
except:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=False, env=env)
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', 'torchvision', 'torchaudio',
                    '--index-url', 'https://download.pytorch.org/whl/cu128'], check=False, env=env)

print('Installing xformers...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'xformers',
                '--index-url', 'https://download.pytorch.org/whl/cu128'], check=False, env=env)

print('Installing Wan2GP requirements...')
req = WAN2GP_ROOT / 'requirements.txt'
if req.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req)], check=False, env=env)

# Fix matplotlib
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
if target.exists():
    text = target.read_text()
    if "matplotlib.use('TkAgg')" in text:
        target.write_text(text.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')", 1))
        print('Fixed matplotlib backend.')

import torch
print(f'\nTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---## Step 6 — Launch Gradio UIClick play, wait for the public URL, then click it.

In [ ]:
import os, subprocess, sys, threading, time
from pathlib import Path

WAN2GP_ROOT = Path('/content/Wan2GP').resolve()
WAN_CACHE_DIR = Path('/content/Wan2GP-data/cache').resolve()

wgp_py = WAN2GP_ROOT / 'wgp.py'
if not wgp_py.exists():
    found = list(WAN2GP_ROOT.rglob('wgp.py'))
    if found: wgp_py = found[0]
    else: raise RuntimeError('wgp.py not found. Re-run Step 5.')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Runtime -> Change runtime type -> GPU.')

env = os.environ.copy()
env['WAN_CACHE_DIR'] = str(WAN_CACHE_DIR)
env['HF_HOME'] = str(WAN_CACHE_DIR / 'huggingface')
env['HUGGINGFACE_HUB_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'hub')
env['TRANSFORMERS_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'transformers')
env['TORCH_HOME'] = str(WAN_CACHE_DIR / 'torch')
env['XDG_CACHE_HOME'] = str(WAN_CACHE_DIR / '.cache')

cmd = [sys.executable, '-u', str(wgp_py), '--listen', '--server-port', '7860', '--share', '--profile', '5']

print('='*60)
print('WAN2GP GRADIO UI')
print('='*60)
print('When the URL appears below:')
print('1. Click the URL to open Gradio')
print('2. Model: Wan 2.2 TextImage2Video 5B FastWan')
print('3. Upload your photo')
print('4. Upload audio from /content/audio/ (seg1a.wav first)')
print('5. Resolution: 480p | Frames: 81 | Generate!')
print('6. Download video, then re-run this cell with next .wav')
print('='*60 + '\n')

process = subprocess.Popen(cmd, cwd=str(WAN2GP_ROOT), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

stop_event = threading.Event()
def keepalive():
    while not stop_event.is_set():
        time.sleep(45)
        if not stop_event.is_set(): print('[keepalive] Still running...')
t = threading.Thread(target=keepalive, daemon=True)
t.start()

try:
    for line in iter(process.stdout.readline, ''):
        if not line: break
        print(line, end='')
except KeyboardInterrupt:
    print('\nStopping...')
    process.terminate()
finally:
    stop_event.set()
    process.wait()
    print(f'Stopped (exit: {process.returncode}).')

---## Step 7 — Download your video

In [ ]:
import glob, os
from pathlib import Path
from google.colab import files

try: _ = WAN_OUTPUTS_DIR
except NameError: WAN_OUTPUTS_DIR = Path('/content/Wan2GP-data/outputs')
try: _ = WAN2GP_ROOT
except NameError: WAN2GP_ROOT = Path('/content/Wan2GP')

search = [str(WAN_OUTPUTS_DIR / '**/*.mp4'), str(WAN2GP_ROOT / 'outputs' / '**/*.mp4'), '/content/**/*.mp4']
videos = []
for p in search: videos.extend(glob.glob(p, recursive=True))
videos = sorted(set(videos), key=os.path.getmtime, reverse=True)

if not videos:
    print('No videos yet. Generate one in Gradio first.')
else:
    print(f'Found {len(videos)} video(s):\n')
    for i, v in enumerate(videos, 1):
        mb = os.path.getsize(v) / (1024*1024)
        print(f'{i}. {v} ({mb:.1f} MB)')
    latest = videos[0]
    print(f'\nDownloading: {latest}')
    files.download(latest)

---## Tips- **Segment order:** seg1a → seg1b → seg1c → seg2 → seg3 → seg4- **Each video:** generate in Gradio → download → re-run Step 6 with next .wav- **Stitch all 6:** Use CapCut (free) or ffmpeg- **Resolution:** 480p for free Colab T4- **Frames:** 81 per segment (~5 seconds at 16fps)